## Builder

---

> **In one line.** A Builder constructs an object as a *composition of small steps*: starting from an empty state $s_0$, each step $g_i$ folds in one change $\delta_i$, and a final projection $\phi$ (`.build()`) reads off the finished object from the accumulated state.

### 1. State space and steps

Fix the **state space** $S$ — the set of all possible object states, every combination of size, crust, toppings, and so on that the thing under construction can be in. Building begins at a distinguished **initial empty state** $s_0 \in S$: a blank object with nothing set yet. After $i$ steps we sit at the **partial state** $s_i \in S$, a half-finished object.

A **builder step** is a function on states. The $i$-th step $g_i$ takes the current state and returns a slightly modified one. The modification it carries is a **small change** $\delta_i$ — "add cheese", "set size to large" — and it is folded in by the **merge operator** $\oplus$:

$$\boxed{\,g_i : S \rightarrow S, \qquad g_i(s) = s \oplus \delta_i\,}$$

Crucially $\oplus$ touches *only* the aspect named by $\delta_i$; every other part of $s$ passes through unchanged.

### 2. Composition into one builder

Running the steps in sequence threads each state into the next,

$$s_1 = g_1(s_0),\quad s_2 = g_2(s_1),\quad \ldots,\quad s_n = g_n(s_{n-1}),$$

which is exactly the **composition** $\circ$ of the step functions — where $g_2 \circ g_1$ means *apply $g_1$ first, then $g_2$*. Collapsing the whole chain gives a single builder map

$$\boxed{\,f \;=\; g_n \circ g_{n-1} \circ \cdots \circ g_2 \circ g_1\,} \qquad f : S \rightarrow S,$$

so that $s_n = f(s_0)$. The data flows one step at a time from the empty frame to the loaded final state:

$$\underbrace{s_0}_{\text{empty}} \;\xrightarrow{\;g_1\;}\; \underbrace{s_1}_{s_0 \oplus \delta_1} \;\xrightarrow{\;g_2\;}\; \underbrace{s_2}_{s_1 \oplus \delta_2} \;\xrightarrow{\;g_3\;}\; \cdots \;\xrightarrow{\;g_n\;}\; \underbrace{s_n}_{f(s_0)}$$

### 3. Projection to the finished object

The accumulated state $s_n$ is still *builder-shaped*. The final step extracts the product: a **projection** $\phi$ maps the loaded state into $\mathcal{O}$, the set of all finished objects in memory. This is the `.build()` call,

$$\boxed{\,\phi : S \rightarrow \mathcal{O}, \qquad \text{result} = \phi(s_n) = \phi\big(f(s_0)\big)\,}$$

and what it returns is the finished product — never the builder itself.

### 4. Key conditions

1. **Incremental change** — each step touches a single aspect and leaves the rest of the state alone:
   $$g_i(s) = s \oplus \delta_i, \qquad \text{all other parts of } s \text{ unchanged.}$$
2. **Order sensitivity** — composition need not commute, so the order of steps can matter:
   $$g_2 \circ g_1 \;\neq\; g_1 \circ g_2 \quad \text{in general.}$$
3. **Projection** — `.build()` is $\phi$, which returns the finished object from the final state, not the builder:
   $$\phi(s_n) \in \mathcal{O}.$$

&nbsp;

> 🏭 Think of an **assembly line**. Each station $g_i$ adds one component to the current state $s_{i-1}$, producing $s_i$. The car starts as an empty frame ($s_0$), gets an engine ($g_1$), wheels ($g_2$), paint ($g_3$). The final inspection $\phi$ signs it off as a complete car.

### Exercise 07 — Pizza Builder

---

**Scenario:** A pizza has many optional parts. Instead of a constructor with 10 arguments, a Builder adds each part as a discrete step $g_i$ applied to the current state $s_{i-1}$.

**Your task:** Build a `PizzaBuilder` with chainable methods. Each method is one $g_i$. `.build()` is $\phi$.

```python
pizza = (PizzaBuilder()          # s_0: empty state
         .set_size("large")      # g_1: s_1 = s_0 ⊕ δ_size
         .set_crust("thin")      # g_2: s_2 = s_1 ⊕ δ_crust
         .add_topping("cheese")  # g_3: s_3 = s_2 ⊕ δ_topping
         .build())               # φ(s_3) → Pizza object
```

**Hints**

- Each method returns `self` — this is how $g_i$ passes the updated state to the next step in the chain.
- `build()` returns a `Pizza` object, not the builder — this is $\phi : S \rightarrow \mathcal{O}$, the projection from state space to finished object.

In [ ]:
# --------------------------------
# Finished object (an element of O) — you do not change this

class Pizza:
    def __init__(self, size, crust, toppings):
        self.size = size
        self.crust = crust
        self.toppings = toppings

    def __repr__(self):
        return f"Pizza(size={self.size!r}, crust={self.crust!r}, toppings={self.toppings})"

# --------------------------------
# Builder — holds the state s and applies steps g_i; .build() is the projection phi

class PizzaBuilder:
    def __init__(self):
        # s_0: the initial empty state
        self._size = None
        self._crust = None
        self._toppings = []

    def set_size(self, size):        # g_1: s_1 = s_0 ⊕ δ_size
        # store size into state, then return self to pass state on
        ...

    def set_crust(self, crust):      # g_2: s_2 = s_1 ⊕ δ_crust
        ...

    def add_topping(self, topping):  # g_3: s_3 = s_2 ⊕ δ_topping
        ...

    def build(self):                 # φ : S → O  (returns a Pizza, not the builder)
        ...

# --------------------------------
pizza = (PizzaBuilder()          # s_0: empty state
         .set_size("large")      # g_1
         .set_crust("thin")      # g_2
         .add_topping("cheese")  # g_3
         .build())               # φ(s_3)
print(pizza)

### Exercise 08 — SQL Query Builder

---

**Scenario:** A SQL query is built from optional parts: SELECT, FROM, WHERE, ORDER BY. Each part is one $g_i$ applied to the query state. `.build()` assembles and returns the final SQL string.

**Your task:** Build a `QueryBuilder` where each method is a step and `.build()` produces the final SQL string.

```python
q = QueryBuilder().select("*").from_table("users").where("age > 18").build()
# SELECT * FROM users WHERE age > 18
```

**Hints**

- Each part of the query is stored in $s$ as an instance variable. `build()` is $\phi$ — it reads $s_n$ and assembles the parts in the correct order.
- Handle optional steps: if `.where()` was never called, $\delta_{\text{where}} = \emptyset$ and `build()` omits the WHERE clause gracefully.

In [ ]:
# --------------------------------
# Builder — each method is a step g_i storing one part of the query state s;
# .build() is the projection phi that assembles s_n into the final SQL string.

class QueryBuilder:
    def __init__(self):
        # s_0: empty query state — each part starts unset (δ = ∅)
        self._select = None
        self._from = None
        self._where = None
        self._order_by = None

    def select(self, columns):       # g_i: s ⊕ δ_select
        # store columns into state, return self
        ...

    def from_table(self, table):     # g_i: s ⊕ δ_from
        ...

    def where(self, condition):      # g_i: s ⊕ δ_where  (optional step)
        ...

    def order_by(self, columns):     # g_i: s ⊕ δ_order  (optional step)
        ...

    def build(self):                 # φ : S → O — read s_n, assemble parts in order,
        # omitting any part whose δ was never applied
        ...

# --------------------------------
q = QueryBuilder().select("*").from_table("users").where("age > 18").build()
print(q)   # SELECT * FROM users WHERE age > 18